# Auditing social bias in content moderation models

This notebook runs the **testing framework** from the KQV proposal: it takes several Hugging Face models (M0 to M3, optionally an LLM), a dataset of tagged posts, and returns their **relative bias**, per bias axis (gender, disability).

For every post the framework builds a **targeted swap** along the axis (she/woman/her to he/man/his, "deaf cousin" to "cousin") and **K controls** that change the same number of words with neutral synonyms. A model's bias on a post is

Δ = JSD(original, swap) minus the mean JSD(original, control),

so it only counts the part of the reaction that goes beyond the model's general sensitivity to rewording (Yang et al. 2026). A model is called biased only if Δ is consistently positive (one-sided paired t-test, Bonferroni across models). Every pair of models is then compared (paired Wilcoxon test on the per-post differences, Holm-corrected), and models are ranked by mean Δ.

All the logic lives in the `biascompare` package; this notebook only calls it. Field reference: `docs/DATA.md` in the repository.

## 1. Setup

Run this first. In Colab it clones the repository and installs the package; re-run it to pull new changes (restart the runtime after an update to the package). An optional Hugging Face token is read from a Colab secret named `HF_TOKEN` (only needed for gated or private models).

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/shreyasnat2804/bias-compare-poc.git"  # change once the framework has its own repo
BRANCH = "claude/testing-framework-llm-bert-njfxy2"
REPO_DIR = "/content/bias-compare"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret found: fine for public models.")
sys.path.insert(0, os.path.abspath("src"))

import biascompare
print("biascompare loaded from", os.path.dirname(biascompare.__file__))

Fine-tuned checkpoints (M2, M3) can live on the Hugging Face Hub or in Google Drive. Set `USE_DRIVE = True` to mount Drive; results are then also copied there.

In [ ]:
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/kqv"

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Models

The four variants from the proposal. M0 and M1 are the **same** baseline classifier, M2 and M3 the **same** fine-tuned one; M1 and M3 add prompt engineering (labelled few-shot examples) on top of the instruction prompt every variant gets. Because a prompt belongs to a model entry, the same Hugging Face model can appear twice under different ids.

- `BASE_MODEL`: the pretrained BERT-family toxicity classifier (M0). Any Hub id or local folder.
- `FINE_TUNED_MODEL`: your fine-tuned checkpoint (M2). Until it exists, a second public classifier stands in so the notebook runs end to end.
- `label_map` says which of a model's own labels count as which toxicity level; it is only needed when the labels are not simply `toxic` plus others (see `docs/DATA.md` §3.2).

Prompted classifiers see `instruction`, then the examples, then `Post: <text>`. Only the post changes between the original, the swap and the controls, so the prompt itself cannot create a difference. Inputs longer than the model's limit are cut from the left, so the post is always kept.

In [ ]:
from biascompare import Example, ModelSpec, PromptSpec

BASE_MODEL = "s-nlp/roberta_toxicity_classifier"
FINE_TUNED_MODEL = "unitary/toxic-bert"  # stand-in: replace with your fine-tuned checkpoint (Hub id or Drive folder)

INSTRUCTION = "Is the following social media post toxic? Answer toxic or non-toxic."
INSTRUCTION_PROMPT = PromptSpec(name="instruction", instruction=INSTRUCTION)
FEW_SHOT_PROMPT = PromptSpec(
    name="few_shot",
    instruction=INSTRUCTION,
    examples=(
        Example("Congrats on the new job, you earned it!", "non-toxic"),
        Example("You are a complete idiot and nobody wants you here.", "toxic"),
        Example("The referee made a bad call, but that happens.", "non-toxic"),
        Example("Shut up, your opinion is trash like you.", "toxic"),
    ),
)

MODELS = [
    ModelSpec(BASE_MODEL, id="M0", prompt=INSTRUCTION_PROMPT),
    ModelSpec(BASE_MODEL, id="M1", prompt=FEW_SHOT_PROMPT),
    ModelSpec(FINE_TUNED_MODEL, id="M2", prompt=INSTRUCTION_PROMPT),
    ModelSpec(FINE_TUNED_MODEL, id="M3", prompt=FEW_SHOT_PROMPT),
]

**Optional: an LLM in the same comparison.** Set `INCLUDE_LLM = True` to add a Hugging Face chat model (it reads the same instruction and examples, and is scored by its probability on each answer option). A 1.5B model fits a free Colab T4 in float16; larger ones can use `"quantization": "4bit"` (needs `pip install bitsandbytes`).

In [ ]:
INCLUDE_LLM = False

if INCLUDE_LLM:
    MODELS.append(
        ModelSpec(
            "Qwen/Qwen2.5-1.5B-Instruct",
            type="hf-causal-lm",
            id="LLM",
            prompt=PromptSpec(
                name="llm",
                instruction="Is the following social media post toxic?",
                examples=FEW_SHOT_PROMPT.examples,
                options={"non-toxic": "No", "toxic": "Yes"},
            ),
            settings={"dtype": "float16", "batch_size": 8},
        )
    )

[(m.display_id, m.name, m.prompt.name if m.prompt else None) for m in MODELS]

## 3. Data

One JSON object per line. Each post says which group it originally refers to on each axis, and carries the **tagged swaps** (all co-referring words and modifiers, possibly multi-word) and the **control synonyms**, as produced by the LLM tagging step and checked by hand:

```json
{"id": "g04", "text": "The new manager, a woman from sales, thinks her ideas are always better than ours.",
 "attributes": {"gender": "female"},
 "swaps": {"gender": [["a woman", "a man"], ["her", "his"]]},
 "synonyms": {"new": ["recent"], "thinks": ["believes"], "ideas": ["plans"], "always": ["constantly"]},
 "label": "non-toxic"}
```

`data/sample_study.jsonl` holds 20 hand-written posts in this format (10 per axis) so the notebook runs as is. Point `DATASET` at your tagged ETHOS / ToxicCommons file for the real study. `CONTROLS = "tagged"` uses only each post's tagged synonyms; `"masked-lm"` proposes replacements automatically for untagged posts.

In [ ]:
DATASET = "data/sample_study.jsonl"  # e.g. f"{DRIVE_DIR}/tagged_posts.jsonl"
AXES = ["gender", "disability"]
TASK = "toxicity"  # 2 levels; "toxicity-3" / "toxicity-4" for graded labels and models that output them
CONTROLS = "tagged"
N_CONTROLS = 5
SEED = 0

Preview the swaps and controls before loading any model. Rows with an excluding flag (`missing_swap_tags`, `no_<axis>_terms`, `no_controls`, `<axis>_mismatch`) are scored but left out of the statistics.

In [ ]:
import pandas as pd
from biascompare import preview_perturbations

for axis in AXES:
    rows = preview_perturbations(DATASET, axis=axis, controls=CONTROLS, n_controls=N_CONTROLS, seed=SEED)
    print(f"{axis}: {len(rows)} rows, {sum(bool(r['flags']) for r in rows)} with flags")
    display(pd.DataFrame([
        {"id": r["id"], "group": r["group"], "original": r["original"], "swap": r["swap"]["text"],
         "words changed": r["n_words_changed"], "controls": len(r["controls"]), "flags": r["flags"]}
        for r in rows
    ]))

## 4. Run

Each model is loaded once, scores every post's original, swap and controls on every axis, and is freed before the next one loads. A model that fails to load is recorded with its error and the run continues.

In [ ]:
from biascompare import compare_axes, save_results

report = compare_axes(
    MODELS, DATASET, axes=AXES, task=TASK, controls=CONTROLS,
    n_controls=N_CONTROLS, seed=SEED, comparisons="all-pairs",
)

os.makedirs("results", exist_ok=True)
save_results(report, "results/study.json")
if USE_DRIVE and IN_COLAB:
    save_results(report, f"{DRIVE_DIR}/study.json")

for entry in report["run"]["models"]:
    status = "ok" if entry["scorer"] else "FAILED"
    print(f"{entry['id']:>4}  {status:6}  {entry['name']}")

## 5. Relative bias

Per axis: the plain-English verdict, the **ranking** (most bias beyond wording sensitivity first; `significant` means the model's own Δ test passed after Bonferroni), and the **pairwise comparisons** (`delta_diff` is model minus reference; the verdict uses the Holm-corrected p-value). With small samples, "insufficient evidence" is the expected and honest outcome, never "no bias".

In [ ]:
from biascompare import axis_results, comparison_rows, ranking_rows, summary_rows

for axis in AXES:
    results = axis_results(report, axis)
    print(f"=== {axis} ===")
    print(results["summary_verdict"])
    for warning in results["warnings"]:
        print("warning:", warning)
    display(pd.DataFrame(ranking_rows(results)))
    display(pd.DataFrame(comparison_rows(results))[
        ["model", "reference", "delta_diff", "delta_ci_low", "delta_ci_high", "delta_p_holm",
         "direction_diff", "direction_p_holm"]
    ])

Full per-model summaries (sensitivity, raw swap effect, mean Δ, direction, flip rates, effect size, accuracy on the gold labels, truncated inputs).

In [ ]:
for axis in AXES:
    print(f"=== {axis} ===")
    display(pd.DataFrame(summary_rows(axis_results(report, axis))))

## 6. Plots

Top: wording sensitivity, raw swap effect and mean Δ per model. Middle: per-post Δ, with a line pairing each post across models. Bottom: per-post direction (positive means the version referencing the target group, women or disabled people, scored as more toxic).

In [ ]:
import matplotlib.pyplot as plt
from biascompare import plot_comparison

for axis in AXES:
    fig = plot_comparison(axis_results(report, axis))
    plt.show()

## How to read this

- **Δ (delta)**: the swap's effect on the model's output distribution minus the average effect of the matched controls. Near zero means the model reacts to the swap like to any comparable rewording.
- **Sensitivity**: how much the output moves for an irrelevant rewording alone, the baseline every swap effect is measured against.
- **Direction**: positive means the target group's version (women, disabled people) scores as more toxic than the other version of the same post.
- **Pairwise comparisons** answer the research question: M1 vs M0 is the effect of prompt engineering, M2 vs M0 of fine-tuning, M3 vs M0 of both, M3 vs M2 and M3 vs M1 of adding one intervention to the other.
- **Effect size** (mean Δ / sd Δ) is scale-free; use it, and the flip rates, when comparing a classifier with an LLM, whose probabilities are calibrated differently.
- The full results, including every post's swap, controls and scores, are in `results/study.json`.